# Open-Meteo Archive API

**Endpoint:** `https://archive-api.open-meteo.com/v1/archive`

**Purpose:** Historical 30-day weather data (temperature, precipitation, wind) at the epicenter.  
Free to use, no API key required.

**Used in:** `services/flink-enrichment/enrichment_job.py` → `fetch_openmeteo()`

In [1]:
!pip install requests --quiet

In [2]:
import requests
import json
from datetime import datetime, timedelta

In [4]:
BASE_URL = "https://archive-api.open-meteo.com/v1/archive"

# Example coordinates: Turkey (seismically active region)
LAT = 38.4
LON = 38.3

end   = datetime.utcnow().date()
start = end - timedelta(days=30)

params = {
    "latitude":   LAT,
    "longitude":  LON,
    "start_date": str(start),
    "end_date":   str(end),
    "daily":      "temperature_2m_mean,precipitation_sum,windspeed_10m_max",
    "timezone":   "UTC",
}

print(f"Time range: {start} to {end}")
resp = requests.get(BASE_URL, params=params, timeout=15)
# print(f"Status: {resp.status_code}")
# print(f"URL:    {resp.url[:120]}...")
data = resp.json()
# print(json.dumps(data, indent=2))

Time range: 2026-04-12 to 2026-05-12


C:\Users\human\AppData\Local\Temp\ipykernel_9576\3872382717.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end   = datetime.utcnow().date()


In [5]:
def fetch_openmeteo(lat: float, lon: float) -> dict:
    """Open-Meteo: 30-day historical weather baseline."""
    try:
        from datetime import timedelta
        end   = datetime.utcnow().date()
        start = end - timedelta(days=30)
        resp  = requests.get(
            "https://archive-api.open-meteo.com/v1/archive",
            params={
                "latitude":   lat,
                "longitude":  lon,
                "start_date": str(start),
                "end_date":   str(end),
                "daily":      "temperature_2m_mean,precipitation_sum,windspeed_10m_max",
                "timezone":   "UTC",
            },
            timeout=15,
        )
        daily = resp.json().get("daily", {})
        temps = [t for t in daily.get("temperature_2m_mean", []) if t is not None]
        precip = [p for p in daily.get("precipitation_sum", []) if p is not None]
        winds = [w for w in daily.get("windspeed_10m_max", []) if w is not None]
        return {
            "temp_avg_30d":      round(sum(temps)  / len(temps),  2) if temps  else None,
            "precipitation_30d": round(sum(precip) / len(precip), 2) if precip else None,
            "wind_speed_avg":    round(sum(winds)  / len(winds),  2) if winds  else None,
        }
    except Exception as e:
        print(f"Open-Meteo fetch failed: {e}", flush=True)
        return {}

fetched_data = fetch_openmeteo(LAT, LON)
print(json.dumps(fetched_data, indent=2))

{
  "temp_avg_30d": 14.47,
  "precipitation_30d": 2.85,
  "wind_speed_avg": 14.28
}


C:\Users\human\AppData\Local\Temp\ipykernel_9576\2103059901.py:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end   = datetime.utcnow().date()


In [ ]:
# Show daily units
print("Units:")
print(json.dumps(data.get("daily_units", {}), indent=2))

# First 5 days
daily = data.get("daily", {})
print("\nFirst 5 days:")
for i in range(min(5, len(daily.get("time", [])))):
    print({
        "date":        daily["time"][i],
        "temp_mean_C": daily["temperature_2m_mean"][i],
        "precip_mm":   daily["precipitation_sum"][i],
        "wind_kmh":    daily["windspeed_10m_max"][i],
    })

In [ ]:
# What fetch_openmeteo() returns (30-day averages)
temps  = [t for t in daily.get("temperature_2m_mean", []) if t is not None]
precip = [p for p in daily.get("precipitation_sum",   []) if p is not None]
winds  = [w for w in daily.get("windspeed_10m_max",   []) if w is not None]

result = {
    "temp_avg_30d":      round(sum(temps)  / len(temps),  2) if temps  else None,
    "precipitation_30d": round(sum(precip) / len(precip), 2) if precip else None,
    "wind_speed_avg":    round(sum(winds)  / len(winds),  2) if winds  else None,
}
print("Output of fetch_openmeteo():")
print(json.dumps(result, indent=2))

In [ ]:
# Simple temperature visualization
times = daily.get("time", [])
temp_series = daily.get("temperature_2m_mean", [])

print("Temperature trend (30 days):")
print(f"{'Date':<12} {'Temp °C':>8}  {'Bar'}")
print("-" * 50)
for date, temp in zip(times, temp_series):
    if temp is not None:
        bar = "#" * max(0, int(temp / 2))
        print(f"{date:<12} {temp:>8.1f}  {bar}")